## Hospital Patient Data Analysis

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [3]:
## 1.	Load the patient dataset and show summary with info().
df = pd.read_csv('Patient_Data.csv')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


In [5]:
df.head()

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45


In [6]:
## 2.	Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].
df[['PatientID','Department','Doctor','BillAmount']]

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN
5,101,Cardiology,Dr. Smith,5000.0


In [14]:
## 3.	Drop administrative columns like ['ReceptionistID', 'CheckInTime'].
df.drop(columns=['ReceptionistID','CheckInTime'],inplace=True)

In [15]:
df.head()

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN


In [16]:
## 4.Use groupby to find total bill amount per department.
df.groupby('Department')['BillAmount'].sum()

Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64

In [17]:
## 5.Remove duplicate patient records based on PatientID.
df.duplicated().sum()

np.int64(1)

In [19]:
df.drop_duplicates(inplace=True)

In [20]:
df.duplicated().sum()

np.int64(0)

In [22]:
## 6.Fill missing BillAmount values with the mean bill amount.
df.isnull().sum()

PatientID     0
Name          0
Department    0
Doctor        0
BillAmount    2
dtype: int64

In [24]:
df['BillAmount'].mean()

np.float64(6233.333333333333)

In [26]:
df.fillna(df['BillAmount'].mean(),inplace=True)
df

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000
1,102,Bob,Neurology,Dr. John,6233.333333
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000
3,104,David,Cardiology,Dr. Smith,6200.000000
4,105,Eva,Dermatology,Dr. Rose,6233.333333


In [27]:
## 7.Merge the billing dataset with patient dataset on PatientID.
df1 = pd.read_csv('Billing_Data.csv')

In [43]:
df1

,PatientID,InsuranceCovered,FinalAmount
0,101,2000,3000
1,102,1500,3500
2,103,2500,5000
3,104,3000,3200
4,105,1000,4000


In [30]:
df2 = pd.merge(df,df1,on='PatientID')
df2

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000,3000
1,102,Bob,Neurology,Dr. John,6233.333333,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.000000,3000,3200
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000,4000


In [36]:
## 8.Concatenate an additional DataFrame that contains new patients for the current week (row-wise).
dict1 = {'PatientID':[106,107,108],'Name':['Abel','Adam','Janaki'],'Department':['Neurology','Dermatology','Cardiology'],
        'Doctor':['Dr.John','Dr.Rose','Dr.Smith'],'BillAmount':[2500,7000,6000]}
dict1

{'PatientID': [106, 107, 108],
 'Name': ['Abel', 'Adam', 'Janaki'],
 'Department': ['Neurology', 'Dermatology', 'Cardiology'],
 'Doctor': ['Dr.John', 'Dr.Rose', 'Dr.Smith'],
 'BillAmount': [2500, 7000, 6000]}

In [37]:
df3 = pd.DataFrame(dict1)

In [38]:
df4 = pd.concat([df,df3],ignore_index=True,axis=0)
df4

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000
1,102,Bob,Neurology,Dr. John,6233.333333
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000
3,104,David,Cardiology,Dr. Smith,6200.000000
4,105,Eva,Dermatology,Dr. Rose,6233.333333
5,106,Abel,Neurology,Dr.John,2500.000000
6,107,Adam,Dermatology,Dr.Rose,7000.000000
7,108,Janaki,Cardiology,Dr.Smith,6000.000000


In [44]:
## 9.Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).
df5 = pd.concat([df4,df1[['InsuranceCovered','FinalAmount']]],axis=1)
df5

,PatientID,Name,Department,Doctor,BillAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.000000,2000.0,3000.0
1,102,Bob,Neurology,Dr. John,6233.333333,1500.0,3500.0
2,103,Charlie,Orthopedics,Dr. Lee,7500.000000,2500.0,5000.0
3,104,David,Cardiology,Dr. Smith,6200.000000,3000.0,3200.0
4,105,Eva,Dermatology,Dr. Rose,6233.333333,1000.0,4000.0
5,106,Abel,Neurology,Dr.John,2500.000000,NaN,NaN
6,107,Adam,Dermatology,Dr.Rose,7000.000000,NaN,NaN
7,108,Janaki,Cardiology,Dr.Smith,6000.000000,NaN,NaN


In [ ]:
df5.fillna(df5[['InsuranceCovered',']])

In [ ]:
# Final cleaned dataset with accurate billing info.
# •	All missing values handled, merged dataset across PatientID.
# •	Ability to perform further analytics on department-wise revenue or doctor performance.